# 🎯 RFM Customer Segmentation

Segmenting customers by purchasing behavior using Recency, Frequency, and Monetary value, then validating the results directly in SQL.

## 1. Calculating RFM Metrics
Computing Recency, Frequency, and Monetary value per customer, using the latest invoice date in the dataset as the snapshot reference point.

In [1]:
import pandas as pd

# قراءة الملف النظيف
df = pd.read_csv("clean_ecommerce_sales_2026.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# تحديد نقطة مرجعية: آخر تاريخ موجود في الداتا كلها (مش تاريخ اليوم الفعلي)
snapshot_date = df['InvoiceDate'].max()

# حساب RFM لكل عميل
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Total_Amount', 'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print(rfm.sort_values('Monetary', ascending=False).head(10))

      CustomerID  Recency  Frequency   Monetary
1689       14646        1         73  280206.02
4201       18102        0         60  259657.30
3728       17450        7         46  194390.79
3008       16446        0          2  168472.50
1879       14911        0        201  143711.17
55         12415       23         21  124914.53
1333       14156        9         55  117210.08
3771       17511        2         31   91062.38
2702       16029       38         63   80850.84
0          12346      325          1   77183.60


## 2. Scoring & Segmentation
Scoring each customer into quartiles across all three RFM dimensions, then classifying them into behavioral segments (Champions, Loyal Customers, Needs Attention, At Risk).

In [2]:
# تقسيم كل مقياس لـ 4 مجموعات (Quartiles)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

# تصنيف العملاء لفئات سلوكية بناءً على الدرجات
def segment_customer(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4:
        return 'Champions'
    elif int(row['R_Score']) >= 3 and int(row['F_Score']) >= 3:
        return 'Loyal Customers'
    elif int(row['R_Score']) <= 2 and int(row['F_Score']) <= 2:
        return 'At Risk'
    else:
        return 'Needs Attention'

rfm['CustomerSegment'] = rfm.apply(segment_customer, axis=1)

print(rfm['CustomerSegment'].value_counts())

CustomerSegment
At Risk            1504
Needs Attention    1311
Loyal Customers     914
Champions           609
Name: count, dtype: int64


In [3]:
rfm.to_csv("customer_rfm_segments.csv", index=False)
print("✅ تم حفظ تحليل RFM بنجاح!")

✅ تم حفظ تحليل RFM بنجاح!


## 3. Upload to SQL & Cross-Validation
Uploading the segmented results to the SQL database and re-querying the segment counts directly from SQL to confirm they exactly match the Python-side calculation.

In [4]:
import sqlite3

# الاتصال بنفس قاعدة البيانات اللي فيها sales_data بالفعل
conn = sqlite3.connect("ecommerce_db.db")

# رفع جدول الـ RFM الجديد لقاعدة البيانات
rfm.to_sql("customer_rfm_segments", conn, if_exists="replace", index=False)

print(f"✅ تم رفع {len(rfm):,} عميل بنجاح إلى جدول customer_rfm_segments في SQL!")

# تحقق سريع: عدد العملاء في كل فئة مباشرة من قاعدة البيانات (بدل pandas)
query_check = """
SELECT CustomerSegment, COUNT(*) AS CustomerCount, ROUND(AVG(Monetary), 2) AS AvgSpending
FROM customer_rfm_segments
GROUP BY CustomerSegment
ORDER BY AvgSpending DESC;
"""

df_check = pd.read_sql_query(query_check, conn)
print("\n📊 التحقق من التوزيع ومتوسط الإنفاق لكل فئة من داخل SQL:")
display(df_check)

conn.close()

✅ تم رفع 4,338 عميل بنجاح إلى جدول customer_rfm_segments في SQL!

📊 التحقق من التوزيع ومتوسط الإنفاق لكل فئة من داخل SQL:


,CustomerSegment,CustomerCount,AvgSpending
0,Champions,609,7501.84
1,Loyal Customers,914,2205.15
2,Needs Attention,1311,1173.01
3,At Risk,1504,508.82
